# Method 6 (ours): Mirror-guided RePaint, training-free adaptive guidance
Option B. No new training: reuse the diffusion model from notebook 05. At every sampling step pull the model's clean-image estimate in the hole toward its mirrored counterpart (only where the mirror is visible), scaled by how symmetric the visible face is. Variants: RePaint (no mirror), fixed guidance, adaptive guidance.

**Code lives in** `src/methods/mirror_diffusion.py`. This notebook only drives it: train / run, evaluate, save results.

Status: method file is a stub; the cells below run once it is implemented.

## 0. Setup

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules and not os.path.exists('DLpastiA'):
    subprocess.run(['git','clone','-b','claude/upbeat-rubin-7at9xc','https://github.com/EricaCleo/DLpastiA.git','DLpastiA'])
    os.chdir('DLpastiA')
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
%pip install -q kagglehub lpips scikit-image torch-fidelity pyyaml tqdm pandas
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(os.getcwd(), '| torch', torch.__version__, '| device:', dev)
if dev == 'cpu': print('NO GPU: on Colab use Runtime > Change runtime type > GPU')

## 1. Data

In [ ]:
# Data (run notebook 00 once first, or this downloads it). Needs env var KAGGLE_API_TOKEN.
from pathlib import Path
if not Path('data/raw').exists():
    !{sys.executable} scripts/download_data.py
from src.data import get_datasets, get_loaders, load_config
cfg = load_config()
train_loader, _ = get_loaders(batch_size=16)
train_ds, test_ds = get_datasets(cfg)
print(len(train_ds), len(test_ds))

## 2. Build the three variants on the same diffusion model

In [ ]:
from src.methods.mirror_diffusion import MirrorGuidedRePaint
fixed    = MirrorGuidedRePaint(lam=0.1, adaptive=False, device=dev); fixed.name = 'repaint_fixed_mirror'
adaptive = MirrorGuidedRePaint(lam=0.1, adaptive=True,  device=dev); adaptive.name = 'repaint_adaptive_mirror'

## 3. Evaluate on one-sided, centre and random square holes

In [ ]:
from src.eval import evaluate, save_result
for m in (fixed, adaptive):
    for mask in ['square', 'one_sided', 'centre']:
        res = evaluate(m, mask, n_images=100, device=dev)
        save_result(m.name, mask, res); print(m.name, mask, res)

## 4. Lambda sweep + head-turn / asymmetric faces

In [ ]:
# sweep lam in [0.02, 0.05, 0.1, 0.2]; check images where symmetry_score is low: adaptive should beat fixed there